# Exploring the Dataset: Intranet Server Auth Log

**Goal:** Understand the structure of `intranet_server/logs/auth.log` (Linux auth/syslog) to design the `auth_events` table.

This notebook walks through:
1. Loading the raw auth log (272 lines of syslog-style `Month Day Time hostname process[pid]: message`)
2. Parsing the format and exploring every field (including message sub-fields for analysis)
3. Building a raw 1:1 DataFrame (`df_raw`) for DDL; keeping parsed `message_*` columns in `df` for field-by-field exploration only
4. Integrating ground truth labels (8 labeled lines, privilege escalation)
5. Mapping to the planned raw table schema with both PostgreSQL and MySQL DDL
6. Checking for 1NF, 2NF, and 3NF violations using the `normalization_rules_sheet.md` checklist

---

**Dataset:** AIT Log Data Set V2.0 — russellmitchell testbed  
**Source:** https://zenodo.org/records/5789064  
**Host:** intranet_server (main attack target, WordPress/intranet)


## 0. Configuration

Set the path to the `russellmitchell/` dataset folder below.

**Default:** Assumes `russellmitchell/` is at the same level as the repo:
```
data-201-group-project/
├── data-201-security-log-analysis/   <-- this repo
│   └── notebooks/                    <-- this notebook is here
└── russellmitchell/                  <-- dataset is here
```

If your dataset is somewhere else, change `DATASET_ROOT` below. We also need the label file path for ground-truth attack lines.


In [ ]:
from pathlib import Path

# --- CHANGE THIS if your dataset is in a different location ---
DATASET_ROOT = Path("..") / ".." / "russellmitchell"
AUTH_LOG = DATASET_ROOT / "gather" / "intranet_server" / "logs" / "auth.log"
LABEL_FILE = DATASET_ROOT / "labels" / "intranet_server" / "logs" / "auth.log"

for name, path in [
    ("Dataset root", DATASET_ROOT),
    ("Auth log", AUTH_LOG),
    ("Label file", LABEL_FILE),
]:
    status = "FOUND" if path.exists() else "MISSING"
    print(f"{name}: {path.resolve()} [{status}]")

## 1. Load Raw Data

The auth log uses syslog-style lines:
```
Month Day HH:MM:SS hostname process[pid]: message
```
Example: `Jan 23 06:25:05 intranet-server CRON[22883]: pam_unix(cron:session): session closed for user root`

Load all lines with 1-based line numbers (to match the label file).


In [ ]:
with open(AUTH_LOG) as f:
    raw_lines = f.readlines()

print(f"Total lines: {len(raw_lines)}")
print()
print("First 5 lines:")
for i, line in enumerate(raw_lines[:5], 1):
    print(f"  [{i}] {line.rstrip()}")
print()
print("Last 3 lines:")
for i, line in enumerate(raw_lines[-3:], len(raw_lines) - 2):
    print(f"  [{i}] {line.rstrip()}")

## 2. Parse the Auth Log Format

### 2.1 Line structure

Each line has: `Month Day Time hostname process[pid]: message`. The **message** part is free-form and often embeds multiple attributes (e.g. sudo: `TTY=... ; PWD=... ; USER=... ; COMMAND=...`). We store the full message as a single blob for the raw table (1NF violation); we also parse out sub-fields into `message_*` columns for field-by-field analysis only. The DDL will match the raw blob, not the parsed columns.


In [ ]:
import re
from datetime import UTC

import pandas as pd

# Syslog has no year; use dataset year
TARGET_YEAR = 2022

# Match: Month Day Time hostname process[pid]: message
LINE_RX = re.compile(
    r"^([A-Z][a-z]{2})\s+(\d{1,2})\s+(\d{2}:\d{2}:\d{2})\s+(\S+)\s+([^:\[]+)(?:\[(\d+)\])?:\s+(.*)$"
)


def parse_auth_line(line_num, line):
    # Parse one auth.log line. Returns dict with line_number, event_timestamp,
    # hostname, process_name, pid, message (raw blob), and message_* for analysis only.
    record = {"line_number": line_num}
    line = line.rstrip()
    m = LINE_RX.match(line)
    if not m:
        return record
    month, day, time_str, hostname, process_name, pid, message = m.groups()
    record["hostname"] = hostname
    record["process_name"] = process_name.strip() if process_name else None
    record["pid"] = int(pid) if pid else None
    record["message"] = message.strip() if message else ""

    # Timestamp
    ts_str = f"{TARGET_YEAR}-{month}-{int(day):02d} {time_str}"
    try:
        record["event_timestamp"] = pd.to_datetime(ts_str, format="%Y-%b-%d %H:%M:%S").tz_localize(
            UTC
        )
    except Exception:
        record["event_timestamp"] = None

    # --- Parsed sub-fields (message_*) for analysis only; do NOT put in raw DDL ---
    # session for user X
    mu = re.search(r"(?:session (?:opened|closed) for user|Accepted \w+ for) (\S+)", message)
    if mu:
        record["message_user"] = mu.group(1)
    # by (uid=N)
    muid = re.search(r"by \((?:uid=)?(\d+)\)", message)
    if muid:
        record["message_uid"] = int(muid.group(1))
    # sshd: from IP port N
    mip = re.search(r"from (\d+\.\d+\.\d+\.\d+) port (\d+)", message)
    if mip:
        record["message_remote_ip"] = mip.group(1)
        record["message_remote_port"] = int(mip.group(2))
    # sshd: Accepted <method> for
    m_auth = re.search(r"Accepted (\w+) for", message)
    if m_auth:
        record["message_auth_method"] = m_auth.group(1)
    # sudo: TTY=... ; PWD=... ; USER=... ; COMMAND=...
    mty = re.search(r"TTY=([^;]+)", message)
    if mty:
        record["message_tty"] = mty.group(1).strip()
    mpwd = re.search(r"PWD=([^;]+)", message)
    if mpwd:
        record["message_pwd"] = mpwd.group(1).strip()
    muser = re.search(r"USER=([^;]+)", message)
    if muser:
        record["message_target_user"] = muser.group(1).strip()
    mcmd = re.search(r"COMMAND=(.+)$", message)
    if mcmd:
        record["message_command"] = mcmd.group(1).strip()
    # su: Successful su for X by Y
    msu = re.search(r"Successful su for (\S+) by (\S+)", message)
    if msu:
        record["message_target_user"] = msu.group(1)
        record["message_actor_user"] = msu.group(2)
    # systemd-logind: New session X of user Y. / Removed session X.
    mses = re.search(r"(?:New session|Removed session) ([^.]+)", message)
    if mses:
        record["message_session_id"] = mses.group(1).strip()

    return record


parsed = [parse_auth_line(i, line) for i, line in enumerate(raw_lines, 1)]
print(f"Parsed {len(parsed)} records")
print(f"Sample (line 1): {parsed[0]}")

In [ ]:
df = pd.DataFrame(parsed)

# pid as nullable int
if "pid" in df.columns:
    df["pid"] = df["pid"].astype("Int64")
if "message_uid" in df.columns:
    df["message_uid"] = df["message_uid"].astype("Int64")
if "message_remote_port" in df.columns:
    df["message_remote_port"] = df["message_remote_port"].astype("Int64")

print(f"Shape: {df.shape}")
print(f"Columns ({len(df.columns)}):")
for col in df.columns:
    non_null = df[col].notna().sum()
    n = len(df)
    uniq = df[col].nunique()
    print(f"  {col:25s}  non-null: {non_null:3d}/{n}  unique: {uniq}")

## 3. Field-by-Field Exploration

### 3.1 process_name (event family)

`process_name` determines which parts of `message` are meaningful (3NF: type -> field set).


In [ ]:
print("=== process_name distribution ===")
proc = df["process_name"].value_counts()
for name, count in proc.items():
    print(f"  {name:20s} {count:4d} ({count / len(df) * 100:5.1f}%)")

### 3.2 event_timestamp


In [ ]:
print("=== event_timestamp range ===")
print(f"  Earliest: {df['event_timestamp'].min()}")
print(f"  Latest:   {df['event_timestamp'].max()}")
print()
print("=== Events per day ===")
days = df["event_timestamp"].dt.date
for d, c in days.value_counts().sort_index().items():
    print(f"  {d}  {c:4d}")

### 3.3 message_* coverage (parsed sub-fields)

These columns are **analysis-only**; they are parsed from the `message` blob and are sparse (process-dependent).


In [ ]:
msg_cols = [c for c in df.columns if c.startswith("message_")]
if msg_cols:
    print("message_* column coverage:")
    for col in msg_cols:
        non_null = df[col].notna().sum()
        print(f"  {col:25s} {non_null:3d}/{len(df)} ({non_null / len(df) * 100:.1f}%)")
else:
    print("No message_* columns present.")

### 3.4 High-signal events (non-CRON)

Filter to interactive/security-relevant processes: sshd, sudo, su, systemd-logind, systemd.


In [ ]:
non_cron = df[df["process_name"] != "CRON"]
print(f"Non-CRON lines: {len(non_cron)}")
print(non_cron[["line_number", "event_timestamp", "process_name", "pid", "message"]].to_string())

## 4. Raw 1:1 DataFrame

Build `df_raw` by dropping all `message_*` columns. The **DDL must match `df_raw`**, not `df`. The raw table stores only the full `message` as a single TEXT blob (1NF violation).


In [ ]:
message_star_cols = [c for c in df.columns if c.startswith("message_")]
df_raw = df.drop(columns=message_star_cols)

print(f"Raw DataFrame shape: {df_raw.shape}")
print(f"Analysis DataFrame shape: {df.shape}")
print(f"Dropped message_* columns: {message_star_cols}")
print(f"Raw columns ({len(df_raw.columns)}): {list(df_raw.columns)}")
print()
print("Null counts (raw):")
for col in df_raw.columns:
    n = df_raw[col].isnull().sum()
    if n > 0:
        print(f"  {col}: {n} ({n / len(df_raw) * 100:.1f}%)")

In [ ]:
print("=== First 5 rows (df_raw) ===")
print(df_raw.head().to_string())
print()
print("=== Last 3 rows ===")
print(df_raw.tail().to_string())

## 5. Ground Truth Labels (optional)

The label file maps line numbers to attack-phase tags. Join to see which lines are labeled.


In [ ]:
import json

labeled_lines = []
if LABEL_FILE.exists():
    with open(LABEL_FILE) as f:
        for entry in f:
            entry = entry.strip()
            if not entry:
                continue
            obj = json.loads(entry)
            labeled_lines.append(obj["line"])
    labeled_lines = sorted(set(labeled_lines))
    print(f"Labeled line numbers: {labeled_lines}")
    df_labeled = df_raw[df_raw["line_number"].isin(labeled_lines)]
    print(f"Matched rows in df_raw: {len(df_labeled)}")
    print(df_labeled[["line_number", "event_timestamp", "process_name", "message"]].to_string())
else:
    print("Label file not found; skipping labels.")

## 6. Schema Mapping

Map the parsed fields to the planned **raw** table `auth_events_raw` (raw 1:1 loading, no normalization). The table stores `message` as a single TEXT blob (1NF violation). Parsed sub-fields (`message_*`) are for analysis only and are **not** in the raw DDL.

### 6.1 Raw field → column mapping

| # | Raw field | DB Column | PostgreSQL Type | MySQL Type | Nullable | Notes |
|---|-----------|-----------|----------------|------------|----------|-------|
| 1 | *(auto)* | `auth_event_id` | `SERIAL PRIMARY KEY` | `INT AUTO_INCREMENT PRIMARY KEY` | No | Surrogate key |
| 2 | (line index) | `line_number` | `INTEGER NOT NULL` | `INT NOT NULL` | No | 1-based; candidate key; joins to labels |
| 3 | syslog timestamp | `event_timestamp` | `TIMESTAMP WITH TIME ZONE` | `DATETIME` | Yes | Parsed from line (no year in syslog; use dataset year) |
| 4 | hostname | `hostname` | `VARCHAR(100)` | `VARCHAR(100)` | Yes | Always `intranet-server` in this file |
| 5 | process name | `process_name` | `VARCHAR(50) NOT NULL` | `VARCHAR(50) NOT NULL` | No | CRON, sshd, sudo, su, systemd-logind, systemd |
| 6 | PID | `pid` | `INTEGER` | `INT` | Yes | From `[pid]` when present (missing on some sudo/systemd lines) |
| 7 | full message | `message` | `TEXT NOT NULL` | `TEXT NOT NULL` | No | Raw blob (1NF violation: embeds TTY, PWD, USER, COMMAND, IP, etc.) |

Optional columns when loading with labels (see `hunt_auth_logs_findings.md`): `auth_event_category` (TEXT[] / JSON), `auth_signature_matches` (JSONB / JSON) store labels and rules as-is; normalization unpacks them into `attack_labels`.

### 6.2 Raw DDL



In [ ]:
postgresql_ddl = """
-- PostgreSQL
CREATE TABLE auth_events_raw (
    auth_event_id   SERIAL PRIMARY KEY,
    line_number     INTEGER NOT NULL,
    event_timestamp TIMESTAMP WITH TIME ZONE,
    hostname        VARCHAR(100),
    process_name    VARCHAR(50) NOT NULL,
    pid             INTEGER,
    message         TEXT NOT NULL,
    created_at      TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
"""

mysql_ddl = """
-- MySQL
CREATE TABLE auth_events_raw (
    auth_event_id   INT AUTO_INCREMENT PRIMARY KEY,
    line_number     INT NOT NULL,
    event_timestamp DATETIME,
    hostname        VARCHAR(100),
    process_name    VARCHAR(50) NOT NULL,
    pid             INT,
    message         TEXT NOT NULL,
    created_at      DATETIME DEFAULT CURRENT_TIMESTAMP
);
"""

print(postgresql_ddl)
print(mysql_ddl)

## 7. Observations for Normalization Phase

Use the `normalization_rules_sheet.md` checklist.

### 7.1 1NF Check

- **Multi-valued / composite field:** The `message` column often contains multiple attributes in one cell (e.g. sudo: `TTY=... ; PWD=... ; USER=... ; COMMAND=...`; sshd: remote IP, port, auth method). **1NF violated.** Resolution: unpack into separate columns during normalization.
- **Repeating groups:** None in this table (no phone1/phone2-style columns).

### 7.2 2NF Check

- The table has a **single-column primary key** (`auth_log_id`). Partial dependencies require a composite key, so **2NF is satisfied**.

### 7.3 3NF Check

- **Transitive dependency:** `process_name` determines which subset of attributes can be parsed from `message` (CRON vs sshd vs sudo vs su vs systemd-logind). So `auth_log_id -> process_name -> populated_field_set`. **3NF violated.** Resolution: subtype tables or type-specific columns in the final 3NF schema.

### Preliminary Functional Dependencies

| FD | Determinant | Dependent(s) | Reasoning |
|----|-------------|---------------|-----------|
| FD1 | `auth_log_id` | all other attributes | Surrogate PK. |
| FD2 | `line_number` | all other attributes | Each log line is unique; candidate key. |
| FD3 | `process_name` | populated field set | Process family determines which message sub-fields exist (3NF violation). |
| FD4 | `hostname` | constant in this file | Single host; meaningful after unioning multiple sources. |


## 8. Summary

### What we discovered

1. `auth.log` has 272 lines; syslog format with `hostname process[pid]: message`.
2. **1NF violated:** `message` is a composite blob (TTY, PWD, USER, COMMAND, IP, etc. embedded).
3. **2NF satisfied:** single-column PK.
4. **3NF violated:** `process_name` determines which fields are meaningful (type → field set).
5. CRON dominates (~94.5%); high-signal events are sshd, sudo, su (15 non-CRON lines). 8 labeled lines (145–152) are privilege escalation (su to jhall, sudo commands).

### Raw schema output

- `df_raw`: 7 columns (line_number, event_timestamp, hostname, process_name, pid, message) — DDL matches this; table name `auth_events_raw`.
- Parsed `message_*` columns live in `df` for analysis only; they are not in the raw table.

### Next steps

- Consolidate findings in `docs/data_exploration/notebook_findings/hunt_auth_logs_findings.md`.
- In normalization: unpack `message` into atomic columns; resolve type → field_set via subtype or derived columns; optionally merge with audit.log for a unified auth_events table.
